<a href="https://colab.research.google.com/github/irezendegarcia-prog/python_sis_26_3_periodo/blob/Italo_Rezende/aula_12/Italo_Rezende_aula_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Atividade Prática 01 - Aula 12: Auditoria Mensal (Servidores e Logs)

In [ ]:
# Etapa 1: Instalação, importações (PEP 8), criação da planilha e inserção inicial
!pip install -q openpyxl

import openpyxl
from openpyxl.styles import Font, PatternFill

wb = openpyxl.Workbook()
ws_servidores = wb.active
ws_servidores.title = "Servidores"

cabecalho = ["Hostname", "IP", "Memória (GB)", "Uso de Disco (%)", "Status"]
ws_servidores.append(cabecalho)

for celula in ws_servidores[1]:
    celula.font = Font(bold=True)

servidores = [
    ["srv-web-01", "192.168.10.10", 16, 45, "Online"],
    ["srv-app-02", "192.168.10.11", 32, 84, "Alerta"],
    ["srv-db-03", "192.168.10.12", 64, 92, "Crítico"],
    ["srv-bkp-04", "192.168.10.13", 32, 62, "Online"],
]

for servidor in servidores:
    ws_servidores.append(servidor)

print(f"Planilha criada com a aba 'Servidores' e {len(servidores)} registros.")


In [ ]:
# Etapa 2: Edição e alerta para servidores com Uso de Disco (%) >= 80%
preenchimento_alerta = PatternFill(
    start_color="FFC7CE",
    end_color="FFC7CE",
    fill_type="solid",
)
fonte_alerta = Font(color="9C0006", bold=True)

for linha in ws_servidores.iter_rows(min_row=2, max_row=ws_servidores.max_row):
    hostname = linha[0].value
    celula_disco = linha[3]
    uso_disco = celula_disco.value

    if uso_disco >= 80:
        celula_disco.fill = preenchimento_alerta
        celula_disco.font = fonte_alerta
        print(
            f"[ALERTA] Servidor '{hostname}' com {uso_disco}% de uso de disco: "
            "precisa de expansão de armazenamento."
        )


In [ ]:
# Geração do arquivo log.txt de referência (Aula 09) para execução autocontida no Colab
with open("log.txt", "w", encoding="utf-8") as log:
    log.write(
        """2026-10-14 08:30:15 - INFO - Sistema inicializado com sucesso
2026-10-14 08:31:02 - INFO - Usuario 'admin' realizou login
2026-10-14 08:35:22 - AVISO - Tentativa de acesso com senha incorreta: usuario 'bat'
2026-10-14 08:35:40 - SUCESSO - Usuario 'bat' realizou login
2026-10-14 08:40:11 - ERRO - Falha ao conectar ao banco de dados: timeout
2026-10-14 08:42:00 - INFO - Conexao reestabelecida
"""
    )

print("Arquivo 'log.txt' pronto no diretório de execução.")


In [ ]:
# Etapa 3: Abertura do log.txt com with/try/except e extração do nível do evento
nome_log = "log.txt"
registros_log = []

try:
    with open(nome_log, "r", encoding="utf-8") as arquivo_log:
        for linha in arquivo_log:
            linha_limpa = linha.strip()
            if not linha_limpa:
                continue

            partes = linha_limpa.split(" - ", 2)
            if len(partes) == 3:
                data_hora, tipo_evento, mensagem = [p.strip() for p in partes]
                registros_log.append((data_hora, tipo_evento, mensagem))
                print(f"Evento extraído -> Nível: {tipo_evento} | Data/Hora: {data_hora}")
except FileNotFoundError:
    print(f"[AVISO] O arquivo '{nome_log}' não foi encontrado no diretório atual.")

print(f"\nTotal de eventos extraídos do log: {len(registros_log)}")


In [ ]:
# Etapa 4: Nova aba para os logs e salvamento em auditoria_logs.xlsx
ws_logs = wb.create_sheet(title="Auditoria de Logs")
ws_logs.append(["Data/Hora", "Tipo de Evento", "Mensagem"])

for celula in ws_logs[1]:
    celula.font = Font(bold=True)

for registro in registros_log:
    ws_logs.append(list(registro))

arquivo_final = "auditoria_logs.xlsx"
wb.save(arquivo_final)
print(
    f"Arquivo '{arquivo_final}' salvo com sucesso contendo as abas: {wb.sheetnames}."
)


## Atividade Prática 02 - Aula 12: Coleta de Métricas de Hardware

In [ ]:
# Etapa 1: Captura instantânea de CPU e memória RAM com psutil
!pip install -q psutil

import time

import psutil

cpu_atual = psutil.cpu_percent(interval=0.5)
ram_atual = psutil.virtual_memory().percent

print(f"Captura instantânea -> CPU: {cpu_atual:.1f}% | RAM: {ram_atual:.1f}%")


In [ ]:
# Etapa 2: Configuração do ciclo de amostragem (5 iterações com pausa de 1 segundo)
total_iteracoes = 5
intervalo_segundos = 1


def capturar_metricas():
    """Retorna uma tupla (uso_cpu_pct, uso_ram_pct)."""
    uso_cpu = psutil.cpu_percent(interval=None)
    uso_ram = psutil.virtual_memory().percent
    return uso_cpu, uso_ram


# Inicializa a leitura base do psutil para evitar leitura zerada na primeira amostra
psutil.cpu_percent(interval=None)
print(
    f"Ciclo configurado: {total_iteracoes} iterações com pausa de {intervalo_segundos}s."
)


In [ ]:
# Etapa 3: Criação do arquivo telemetria_sistema.csv com cabeçalho explícito
caminho_csv = "telemetria_sistema.csv"

with open(caminho_csv, "w", encoding="utf-8") as arquivo_csv:
    arquivo_csv.write("Iteracao,Uso_CPU_Pct,Uso_RAM_Pct\n")

print(f"Arquivo '{caminho_csv}' inicializado com o cabeçalho.")


In [ ]:
# Etapa 4: Coleta das 5 amostras e persistência incremental em modo append ('a')
for iteracao in range(1, total_iteracoes + 1):
    time.sleep(intervalo_segundos)
    uso_cpu, uso_ram = capturar_metricas()

    with open(caminho_csv, "a", encoding="utf-8") as arquivo_csv:
        arquivo_csv.write(f"{iteracao},{uso_cpu:.2f},{uso_ram:.2f}\n")

    print(f"Iteração {iteracao}/{total_iteracoes} gravada -> CPU: {uso_cpu:.2f}% | RAM: {uso_ram:.2f}%")

print(f"\nColeta finalizada e salva em '{caminho_csv}'.")


## Atividade Prática 03 - Aula 12: Análise e Visualização de Métricas

In [ ]:
# Etapa 1: Ingestão do arquivo telemetria_sistema.csv com pandas
import matplotlib.pyplot as plt
import pandas as pd

df_telemetria = pd.read_csv("telemetria_sistema.csv")
print("Dados carregados de 'telemetria_sistema.csv':")
print(df_telemetria)


In [ ]:
# Etapa 2: Cálculo das médias aritméticas e resumo descritivo no console
media_cpu = df_telemetria["Uso_CPU_Pct"].mean()
media_ram = df_telemetria["Uso_RAM_Pct"].mean()

print("=== RESUMO DESCRITIVO DE TELEMETRIA ===")
print(f"Total de amostras: {len(df_telemetria)}")
print(f"Média de uso de CPU: {media_cpu:.2f}%")
print(f"Média de uso de RAM: {media_ram:.2f}%")
print("\nEstatísticas detalhadas:")
print(df_telemetria[["Uso_CPU_Pct", "Uso_RAM_Pct"]].describe().round(2))


In [ ]:
# Etapa 3: Gráfico de linha com matplotlib (CPU e RAM ao longo das iterações)
plt.figure(figsize=(8, 4.5))
plt.plot(
    df_telemetria["Iteracao"],
    df_telemetria["Uso_CPU_Pct"],
    marker="o",
    linewidth=2,
    label="Uso de CPU (%)",
)
plt.plot(
    df_telemetria["Iteracao"],
    df_telemetria["Uso_RAM_Pct"],
    marker="s",
    linewidth=2,
    label="Uso de RAM (%)",
)

plt.title("Variação do Uso de CPU e RAM ao Longo das Iterações")
plt.xlabel("Iteração")
plt.ylabel("Uso (%)")
plt.xticks(df_telemetria["Iteracao"])
plt.ylim(0, 100)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()
